In [0]:
from pyspark.sql.functions import current_timestamp

spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.bronze")

tabelas_bronze = {
    "olist_customers_dataset.csv": "tb_customers",
    "olist_geolocation_dataset.csv": "tb_geolocalizacao",
    "olist_order_items_dataset.csv": "tb_order_items",
    "olist_order_payments_dataset.csv": "tb_order_payments",
    "olist_order_reviews_dataset.csv": "tb_order_reviews",
    "olist_orders_dataset.csv": "tb_orders",
    "olist_products_dataset.csv": "tb_products",
    "olist_sellers_dataset.csv": "tb_sellers",
    "product_category_name_translation.csv": "tb_product_category_name_translation"
}

caminho_landing_zone = "/Volumes/workspace/default/landing_zone/"

for arquivo, tabela in tabelas_bronze.items():
    print(f"Processando {arquivo} para a tabela workspace.bronze.{tabela}...")
    
    df = spark.read.csv(
        f"{caminho_landing_zone}{arquivo}",
        header=True,
        inferSchema=True,
        escape='"'
    )
    
    df_com_timestamp = df.withColumn("timestamp_ingestion", current_timestamp())
    
    df_com_timestamp.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(f"workspace.bronze.{tabela}")
        
print("Ingestão concluida")

Processando olist_customers_dataset.csv para a tabela workspace.bronze.tb_customers...
Processando olist_geolocation_dataset.csv para a tabela workspace.bronze.tb_geolocalizacao...
Processando olist_order_items_dataset.csv para a tabela workspace.bronze.tb_order_items...
Processando olist_order_payments_dataset.csv para a tabela workspace.bronze.tb_order_payments...
Processando olist_order_reviews_dataset.csv para a tabela workspace.bronze.tb_order_reviews...
Processando olist_orders_dataset.csv para a tabela workspace.bronze.tb_orders...
Processando olist_products_dataset.csv para a tabela workspace.bronze.tb_products...
Processando olist_sellers_dataset.csv para a tabela workspace.bronze.tb_sellers...
Processando product_category_name_translation.csv para a tabela workspace.bronze.tb_product_category_name_translation...
Ingestão concluida


In [0]:
import requests
from pyspark.sql.types import StructType, StructField, StringType, DoubleType
from pyspark.sql.functions import current_timestamp

data_inicio_formatada = "01-01-2016" 
data_fim_formatada = "12-31-2018"

print("Acessando a API do Banco Central")

url = f"https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/CotacaoDolarPeriodo(dataInicial=@dataInicial,dataFinalCotacao=@dataFinalCotacao)?@dataInicial='{data_inicio_formatada}'&@dataFinalCotacao='{data_fim_formatada}'&$select=dataHoraCotacao,cotacaoCompra&$format=json"

response = requests.get(url)
dados_api = response.json()['value']

schema = StructType([
    StructField("dataHoraCotacao", StringType(), True),
    StructField("cotacaoCompra", DoubleType(), True)
])

df_dolar = spark.createDataFrame(dados_api, schema=schema)

df_dolar_timestamp = df_dolar.withColumn("timestamp_ingestion", current_timestamp())

df_dolar_timestamp.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.bronze.tb_cotacao_dolar")

print("Tabela workspace.bronze.tb_cotacao_dolar gerada.")

Acessando a API do Banco Central
Tabela workspace.bronze.tb_cotacao_dolar gerada.
